# Code Burst Edition: Spec-Driven Row-Wise Workflow

This notebook is a simplified workshop version of the spec-driven row-wise workflow pattern.

The goal is to help participants:
1. Load a small sample dataset or approved bring-your-own dataset.
2. Define a bounded review/classification/extraction task.
3. Send each row through a structured LLM prompt.
4. Parse the response into reviewable fields.
5. Export results for human review.

## Data Use Guardrails

For this workshop, only use data that is:
- Synthetic
- Publicly shareable
- De-identified
- Approved for this sandbox environment

Do **not** upload or process:
- PHI
- Sensitive personal data
- Credentials, API keys, or tokens
- Restricted datasets
- Private grant, personnel, or administrative data unless explicitly approved

## Notebook Philosophy

This notebook is intentionally simplified for a live mini hackathon.  
Instead of using a separate config file, editable settings are placed directly in the notebook so participants can see and modify the workflow more easily.

In [ ]:
# ============================================================
# Code Burst Edition: Notebook Setup + Workshop Controls
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import os
import sys

import pandas as pd

# ------------------------------------------------------------
# 1. Workshop run controls
# ------------------------------------------------------------
# Keep the first run small. Participants can increase this later.
MAX_ROWS_TO_RUN = 5

# Use "sample" for the provided sample dataset.
# Use "byod" if a participant brings their own approved dataset.
DATA_MODE = "sample"  # options: "sample", "byod"

# ------------------------------------------------------------
# 2. File paths
# ------------------------------------------------------------
# Preferred repo-relative path for GitHub / Azure / shared environments
SAMPLE_DATA_PATH = Path("data/publicationslist_HEAL.xlsx")

# Participants can update this if using BYOD mode.
BYOD_DATA_PATH = Path("data/byod/your_byod_dataset.xlsx")

# Output folders
OUTPUT_DIR = Path("outputs")
LOG_DIR = Path("logs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for outputs
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# ------------------------------------------------------------
# 3. Data guardrail acknowledgement
# ------------------------------------------------------------
# For the workshop, this should remain True only if the dataset is approved
# for use in the sandbox environment.
DATA_USE_APPROVED = True

if not DATA_USE_APPROVED:
    raise ValueError(
        "Data use has not been approved. Please confirm the dataset is synthetic, public, "
        "de-identified, or otherwise approved for this sandbox."
    )

# ------------------------------------------------------------
# 4. Select active data path
# ------------------------------------------------------------
if DATA_MODE == "sample":
    DATA_PATH = SAMPLE_DATA_PATH

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "Could not find the sample dataset. Expected it at:\n"
            f"{SAMPLE_DATA_PATH.resolve()}\n\n"
            "Make sure the sample file is committed to the repo under the data/ folder."
        )

elif DATA_MODE == "byod":
    DATA_PATH = BYOD_DATA_PATH

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            "Could not find the BYOD dataset. Please update BYOD_DATA_PATH with a repo-relative "
            "path or a local path that is not committed to the public repo."
        )

else:
    raise ValueError("DATA_MODE must be either 'sample' or 'byod'.")

# ------------------------------------------------------------
# 5. Print setup summary
# ------------------------------------------------------------
print("Notebook setup complete.")
print(f"Python version: {sys.version.split()[0]}")
print(f"Data mode: {DATA_MODE}")
print(f"Data path: {DATA_PATH}")
print(f"Max rows to run: {MAX_ROWS_TO_RUN}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Log folder: {LOG_DIR.resolve()}")
print(f"Run timestamp: {RUN_TIMESTAMP}")

## Select an AI Provider

This workflow is designed to be provider-agnostic at the workflow level.

The row-wise logic stays the same:
1. Build structured row context
2. Send the row to an LLM
3. Request a constrained JSON response
4. Parse and validate the output
5. Export reviewable results

For this workshop, the available models and providers may depend on what is provisioned in the Azure sandbox environment.  
The current default is OpenAI, but this cell is intentionally separated so the model provider can be changed later without rewriting the rest of the notebook.

In [ ]:
# ============================================================
# Add sandbox API key
# ============================================================

# ------------------------------------------------------------
# 1. Add the API key provided for the workshop sandbox
# ------------------------------------------------------------
# Public/GitHub-safe version:
# Leave the placeholder in the committed notebook.
# During the workshop, replace the placeholder with the sandbox API key.
API_KEY = "PASTE_YOUR_API_KEY_HERE"

# ------------------------------------------------------------
# 2. Validate key
# ------------------------------------------------------------
if not API_KEY or API_KEY.strip() == "":
    raise ValueError(
        "API_KEY is blank. Please paste the workshop/sandbox API key before running the notebook."
    )

if API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise ValueError(
        "API_KEY is still set to the placeholder value. "
        "Please replace it with the workshop/sandbox API key before running the notebook."
    )

# ------------------------------------------------------------
# 3. Safe setup summary
# ------------------------------------------------------------
print("API key configured successfully.")
print("API_KEY found: Yes")
print("Key preview: hidden for public notebook safety")

## Initialize the AI Client

Now that the workshop/sandbox API key has been added, this cell initializes the AI client and runs a small test call.

This is only a connection check.  
If this cell runs successfully, the notebook is ready to use the model for row-wise evaluation later.

For the Code Burst workshop, the model or deployment name is defined directly in the notebook for simplicity.  
Depending on the sandbox setup, participants may need to update the model name, deployment name, endpoint, or provider-specific client settings.

In [ ]:
# ============================================================
# Initialize AI client + run a small connection test
# ============================================================

from openai import OpenAI

# ------------------------------------------------------------
# 1. Choose model or deployment
# ------------------------------------------------------------
# For the workshop, keep this visible and easy to change.
# Update this value if your sandbox provides a different model or deployment name.
MODEL_NAME = "gpt-4.1-mini"

# ------------------------------------------------------------
# 2. Initialize client
# ------------------------------------------------------------
# This starter version assumes an OpenAI-compatible client.
# If the sandbox uses Azure OpenAI or another provider, this is the main
# section to swap out while keeping the rest of the workflow pattern intact.
client = OpenAI(
    api_key=API_KEY,
    timeout=60.0,
    max_retries=2
)

# ------------------------------------------------------------
# 3. Run a small smoke test
# ------------------------------------------------------------
try:
    test_response = client.responses.create(
        model=MODEL_NAME,
        instructions=(
            "You are a connection test assistant. "
            "Reply with exactly this phrase: API connection successful."
        ),
        input="Run a connection test.",
        max_output_tokens=20
    )

    print("AI client initialized successfully.")
    print(f"Model selected: {MODEL_NAME}")
    print(f"Test response: {test_response.output_text}")

except Exception as e:
    print("AI client connection test failed.")
    print("Error details:")
    print(e)

In [ ]:
# ============================================================
# Define reusable AI model call helper
# ============================================================

def call_ai_model(
    instructions,
    prompt,
    max_output_tokens=500,
    temperature=0,
    model_name=MODEL_NAME,
):
    """
    Send one prompt to the configured AI model and return a simple result tuple.

    This helper keeps provider-specific API logic in one place. If the sandbox
    uses a different model provider later, this is one of the main functions to update.

    Returns:
        raw_output (str): Model response text.
        model_call_success (bool): Whether the API call completed.
        model_error (str): Error message if the call failed.
    """
    try:
        response = client.responses.create(
            model=model_name,
            instructions=instructions,
            input=prompt,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )

        raw_output = getattr(response, "output_text", "").strip()
        model_call_success = True
        model_error = ""

    except Exception as e:
        raw_output = ""
        model_call_success = False
        model_error = str(e)

    return raw_output, model_call_success, model_error


print("Reusable AI model call helper is ready.")

## Load and Preview the Dataset

Now that the API connection is working, we will load the dataset selected in the setup cell.

This step helps us inspect the structure of the data before building the prompt.  
Before asking the model to evaluate anything, we need to understand:

1. What each row represents
2. Which columns provide useful context
3. Which column should serve as the row identifier
4. Which columns should be included in the LLM prompt
5. Whether the dataset has missing or unexpected values

For the sample workflow, we are using a HEAL publications dataset.  
For BYOD mode, participants can use this same cell to inspect their own approved dataset.

In [ ]:
# ============================================================
# Load and preview dataset
# ============================================================

from IPython.display import display

# ------------------------------------------------------------
# 1. Confirm the selected data path exists
# ------------------------------------------------------------
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the selected data file:\n{DATA_PATH}\n\n"
        "Check DATA_MODE and the file path defined in the setup cell."
    )

print(f"Loading data from: {DATA_PATH}")

# ------------------------------------------------------------
# 2. Load CSV or Excel file
# ------------------------------------------------------------
file_suffix = DATA_PATH.suffix.lower()

if file_suffix in [".xlsx", ".xls"]:
    # Show available sheet names for transparency
    excel_file = pd.ExcelFile(DATA_PATH)
    print("Available Excel sheets:")
    for sheet in excel_file.sheet_names:
        print(f"  - {sheet}")

    # For workshop simplicity, default to the first sheet.
    # Participants can update this if their file has multiple sheets.
    SHEET_NAME = 0

    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

elif file_suffix == ".csv":
    df = pd.read_csv(DATA_PATH)

else:
    raise ValueError(
        "Unsupported file type. Please use a .csv, .xlsx, or .xls file."
    )

# ------------------------------------------------------------
# 3. Remove fully empty rows and columns
# ------------------------------------------------------------
starting_shape = df.shape

df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")

ending_shape = df.shape

# ------------------------------------------------------------
# 4. Print dataset summary
# ------------------------------------------------------------
print("\nDataset loaded successfully.")
print(f"Original shape: {starting_shape[0]} rows x {starting_shape[1]} columns")
print(f"Cleaned shape:  {ending_shape[0]} rows x {ending_shape[1]} columns")

print("\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

# ------------------------------------------------------------
# 5. Preview first few rows
# ------------------------------------------------------------
print("\nPreview of first 5 rows:")
display(df.head())

# ------------------------------------------------------------
# 6. Quick missingness summary
# ------------------------------------------------------------
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column_name", 0: "missing_values"})
)

missing_summary["percent_missing"] = (
    missing_summary["missing_values"] / len(df) * 100
).round(1)

print("\nMissingness summary:")
display(missing_summary)

## Map Columns and Build Row Context

For this sample workflow, each row represents one publication.

We will use:

- `PMID` as the row identifier
- `Title` as LLM context
- `Abstract` as LLM context

The `PMID` will be preserved as the key for saving and merging results later, but only the title and abstract will be sent to the model for evaluation.

This keeps the LLM prompt focused while maintaining traceability back to the original dataset.

In [ ]:
# ============================================================
# Map columns and build row-level context for the sample dataset
# ============================================================

# ------------------------------------------------------------
# 1. Define required columns
# ------------------------------------------------------------
ROW_ID_COLUMN = "PMID"

LLM_CONTEXT_COLUMNS = [
    "Title",
    "Abstract",
]

# ------------------------------------------------------------
# 2. Check that required columns exist
# ------------------------------------------------------------
required_columns = [ROW_ID_COLUMN] + LLM_CONTEXT_COLUMNS

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from the dataset:\n"
        + "\n".join(f"- {col}" for col in missing_columns)
    )

# ------------------------------------------------------------
# 3. Create a working copy
# ------------------------------------------------------------
df_work = df.copy()

# ------------------------------------------------------------
# 4. Helper function to clean cell values
# ------------------------------------------------------------
def clean_cell_value(value):
    """
    Convert cell values into safe, readable strings for prompt context.
    """
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    return str(value).strip()

# ------------------------------------------------------------
# 5. Build structured row context
# ------------------------------------------------------------
def build_row_context(row):
    """
    Build the row payload for the LLM.

    The PMID is retained as the row identifier, but only Title and Abstract
    are included in the LLM context.
    """
    row_id = clean_cell_value(row[ROW_ID_COLUMN])

    llm_context = {}

    for col in LLM_CONTEXT_COLUMNS:
        value = clean_cell_value(row[col])

        if value:
            llm_context[col] = value

    return {
        "row_id": row_id,
        "llm_context": llm_context,
    }

# ------------------------------------------------------------
# 6. Limit to a small prototype set
# ------------------------------------------------------------
prototype_df = df_work.head(MAX_ROWS_TO_RUN).copy()

row_contexts = []

for source_index, row in prototype_df.iterrows():
    row_contexts.append(
        {
            "source_index": source_index,
            **build_row_context(row),
        }
    )

# ------------------------------------------------------------
# 7. Print mapping summary
# ------------------------------------------------------------
print("Column mapping complete.")
print(f"Row ID column: {ROW_ID_COLUMN}")
print("Columns sent to LLM:")
for col in LLM_CONTEXT_COLUMNS:
    print(f"  - {col}")

print(f"\nPrototype rows prepared: {len(row_contexts)}")

# ------------------------------------------------------------
# 8. Preview one row context
# ------------------------------------------------------------
print("\nExample row context:")
print(json.dumps(row_contexts[0], indent=2, ensure_ascii=False))

## Load Publication Type Definitions

The publication type definitions are stored in a separate Markdown file:

`docs/definitions_publication-type.md`

This file acts as a lightweight knowledge source for the workflow.  
Instead of hard-coding the category definitions directly into the notebook, we load them from the file and store them in a Python variable.

Later, this definitions text will be included in the LLM instructions so the model uses the human-defined categories when classifying each publication.

In [ ]:
# ============================================================
# Load publication type definitions
# ============================================================

# ------------------------------------------------------------
# 1. Define possible repo-relative paths to the definitions file
# ------------------------------------------------------------
DEFINITIONS_PATH_OPTIONS = [
    # Preferred repo-relative path when running from the repo root
    Path("docs/definitions_publication-type.md"),

    # Useful if the notebook is running from inside a notebooks/ or src/ folder
    Path("../docs/definitions_publication-type.md"),
]

# ------------------------------------------------------------
# 2. Find the first path that exists
# ------------------------------------------------------------
DEFINITIONS_PATH = None

for path_option in DEFINITIONS_PATH_OPTIONS:
    if path_option.exists():
        DEFINITIONS_PATH = path_option
        break

if DEFINITIONS_PATH is None:
    searched_paths = "\n".join(f"- {path}" for path in DEFINITIONS_PATH_OPTIONS)
    raise FileNotFoundError(
        "Could not find the publication type definitions file. "
        "Checked the following repo-relative paths:\n"
        f"{searched_paths}\n\n"
        "Make sure docs/definitions_publication-type.md exists in the repository."
    )

# ------------------------------------------------------------
# 3. Load definitions text
# ------------------------------------------------------------
PUBLICATION_TYPE_DEFINITIONS = DEFINITIONS_PATH.read_text(encoding="utf-8").strip()

if not PUBLICATION_TYPE_DEFINITIONS:
    raise ValueError(
        f"The definitions file was found, but it appears to be empty:\n{DEFINITIONS_PATH}"
    )

# ------------------------------------------------------------
# 4. Define allowed output labels for later validation
# ------------------------------------------------------------
ALLOWED_PUBLICATION_TYPES = [
    "Original Research Article",
    "Review Article",
    "Clinical Case Report",
    "Research Design and Methods",
    "Opinion Piece",
    "Clinical Practice Guidelines",
    "Systematic Review and Meta-Analysis",
    "Letter",
    "N/A",
]

ALLOWED_CONFIDENCE_LEVELS = [
    "High",
    "Medium",
    "Low",
]

# ------------------------------------------------------------
# 5. Print summary
# ------------------------------------------------------------
print("Publication type definitions loaded successfully.")
print(f"Definitions path: {DEFINITIONS_PATH}")
print(f"Definition text length: {len(PUBLICATION_TYPE_DEFINITIONS):,} characters")

print("\nAllowed publication type labels:")
for label in ALLOWED_PUBLICATION_TYPES:
    print(f"  - {label}")

print("\nAllowed confidence levels:")
for label in ALLOWED_CONFIDENCE_LEVELS:
    print(f"  - {label}")

print("\nDefinitions preview:")
print(PUBLICATION_TYPE_DEFINITIONS[:1000])

## Build the Task Instructions and Row Prompt

Now that the publication type definitions have been loaded, we will build the instructions that will be sent to the model.

This cell combines:

- The publication type definitions
- The allowed output fields
- The rule that the model should only use the title and abstract
- A structured JSON response format

The `PMID` is still retained outside the model call as the row identifier.  
Only the `Title` and `Abstract` are included in the row prompt sent to the model.

This step helps us preview the prompt before running the row-wise classification.

In [ ]:
# ============================================================
# Build task instructions and row prompt
# ============================================================

# ------------------------------------------------------------
# 1. Define the expected JSON output structure
# ------------------------------------------------------------
EXPECTED_OUTPUT_FIELDS = {
    "publication_type": "One allowed publication type label.",
    "confidence": "High, Medium, or Low confidence in the publication type classification.",
    "rationale": "A brief explanation based only on the title and abstract."
}

# ------------------------------------------------------------
# 2. Build task instructions for the model
# ------------------------------------------------------------
TASK_INSTRUCTIONS = f"""
You are assisting with a publication type classification task.

Your job is to review the title and abstract of a publication and classify the article
into one publication type using the definitions provided below.

Use only the title and abstract provided in the row.
Do not use outside knowledge.
Do not infer details that are not supported by the title or abstract.

Publication type definitions:
{PUBLICATION_TYPE_DEFINITIONS}

Allowed publication type labels:
{json.dumps(ALLOWED_PUBLICATION_TYPES, indent=2)}

Allowed confidence levels:
{json.dumps(ALLOWED_CONFIDENCE_LEVELS, indent=2)}

Confidence guidance:
- Use "High" when the title and abstract provide clear, direct evidence for one publication type.
- Use "Medium" when the title and abstract provide some evidence, but there is minor ambiguity.
- Use "Low" when the title and abstract are vague, missing, conflicting, or difficult to classify.

Return your answer as a single valid JSON object with exactly these fields:
{{
  "publication_type": "...",
  "confidence": "...",
  "rationale": "..."
}}

Do not include markdown.
Do not include extra text before or after the JSON.
"""

# ------------------------------------------------------------
# 3. Build the row-level prompt
# ------------------------------------------------------------
def build_publication_prompt(row_payload):
    """
    Build the user-facing prompt for one publication row.

    The PMID is intentionally not included in the prompt content.
    It is retained outside the model call for merging results later.
    """
    llm_context = row_payload["llm_context"]

    title = llm_context.get("Title", "")
    abstract = llm_context.get("Abstract", "")

    prompt = f"""
Classify the publication type based on the title and abstract below.

Title:
{title}

Abstract:
{abstract}
"""

    return prompt.strip()

# ------------------------------------------------------------
# 4. Preview the instructions and first row prompt
# ------------------------------------------------------------
example_row = row_contexts[0]
example_prompt = build_publication_prompt(example_row)

print("Task instructions created successfully.")
print(f"Instruction length: {len(TASK_INSTRUCTIONS):,} characters")

print("\nExpected output fields:")
for field, description in EXPECTED_OUTPUT_FIELDS.items():
    print(f"  - {field}: {description}")

print("\nExample row ID retained outside model call:")
print(example_row["row_id"])

print("\nExample prompt sent to model:")
print("-" * 80)
print(example_prompt)
print("-" * 80)

## Test the LLM Call on One Publication

Before running the workflow across multiple rows, we will test the model on one publication.

This cell sends one row prompt to the model, receives the structured JSON response, parses it, and checks whether the output follows the expected schema.

This helps confirm that:

- The model call works
- The response can be parsed as JSON
- The publication type is one of the allowed labels
- The confidence value is one of the allowed confidence levels
- The rationale field is present

Testing one row first keeps the workflow easier to debug before running the full prototype set.

In [ ]:
# ============================================================
# Test LLM call on one publication row
# ============================================================

# ------------------------------------------------------------
# 1. Helper function to parse model JSON
# ------------------------------------------------------------
def parse_model_json_response(response_text):
    """
    Parse the model response as JSON.

    The prompt asks for a clean JSON object, but this function includes
    a small cleanup step in case markdown code fences appear.
    """
    cleaned_text = response_text.strip()

    if cleaned_text.startswith("```json"):
        cleaned_text = cleaned_text.replace("```json", "", 1).strip()

    if cleaned_text.startswith("```"):
        cleaned_text = cleaned_text.replace("```", "", 1).strip()

    if cleaned_text.endswith("```"):
        cleaned_text = cleaned_text[:-3].strip()

    return json.loads(cleaned_text)


# ------------------------------------------------------------
# 2. Helper function to validate parsed output
# ------------------------------------------------------------
def validate_classification_output(parsed_output):
    """
    Check whether the parsed model output follows the expected structure
    and uses only allowed labels.
    """
    validation_errors = []

    required_fields = [
        "publication_type",
        "confidence",
        "rationale",
    ]

    for field in required_fields:
        if field not in parsed_output:
            validation_errors.append(f"Missing required field: {field}")

    publication_type = parsed_output.get("publication_type")
    confidence = parsed_output.get("confidence")
    rationale = parsed_output.get("rationale")

    if publication_type not in ALLOWED_PUBLICATION_TYPES:
        validation_errors.append(
            f"Invalid publication_type: {publication_type}"
        )

    if confidence not in ALLOWED_CONFIDENCE_LEVELS:
        validation_errors.append(
            f"Invalid confidence: {confidence}"
        )

    if rationale is None or str(rationale).strip() == "":
        validation_errors.append("Rationale is missing or blank.")

    return validation_errors


# ------------------------------------------------------------
# 3. Select one row for testing
# ------------------------------------------------------------
test_row = row_contexts[0]
test_prompt = build_publication_prompt(test_row)

print(f"Testing row ID: {test_row['row_id']}")


# ------------------------------------------------------------
# 4. Call the model using the reusable helper
# ------------------------------------------------------------
raw_output, model_call_success, model_error = call_ai_model(
    instructions=TASK_INSTRUCTIONS,
    prompt=test_prompt,
    max_output_tokens=500,
    temperature=0,
)

if not model_call_success:
    raise RuntimeError(
        "The model call failed. Check the API connection, model name, and prompt.\n"
        f"Model error: {model_error}"
    )


# ------------------------------------------------------------
# 5. Parse and validate the response
# ------------------------------------------------------------
try:
    parsed_output = parse_model_json_response(raw_output)
    parse_success = True
    parse_error = ""

except Exception as e:
    parsed_output = {}
    parse_success = False
    parse_error = str(e)

validation_errors = []

if parse_success:
    validation_errors = validate_classification_output(parsed_output)


# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------
print("\nRaw model output:")
print(raw_output)

print("\nModel call success:")
print(model_call_success)

if model_error:
    print("\nModel error:")
    print(model_error)

print("\nParse success:")
print(parse_success)

if parse_error:
    print("\nParse error:")
    print(parse_error)

print("\nParsed output:")
print(json.dumps(parsed_output, indent=2, ensure_ascii=False))

print("\nValidation errors:")
if validation_errors:
    for error in validation_errors:
        print(f"  - {error}")
else:
    print("  None. Output passed validation.")


# ------------------------------------------------------------
# 7. Store test result for inspection
# ------------------------------------------------------------
test_result = {
    "PMID": test_row["row_id"],
    "raw_model_output": raw_output,
    "model_call_success": model_call_success,
    "model_error": model_error,
    "parse_success": parse_success,
    "parse_error": parse_error,
    "validation_errors": validation_errors,
    **parsed_output,
}

print("\nTest result object:")
print(json.dumps(test_result, indent=2, ensure_ascii=False))

## Run Publication Type Classification on Prototype Rows

Now that the one-row test worked, we can run the same classification workflow across the prototype set.

For each publication row, this cell will:

1. Build the row-level prompt from the `Title` and `Abstract`
2. Send the prompt to the model using the publication type definitions
3. Parse the model response as JSON
4. Validate the output against the allowed labels
5. Store the results in a reviewable dataframe

The `PMID` is retained as the row identifier so the model output can be merged back to the original publication dataset later.

### Prototype Row Limit

By default, this notebook only processes the number of rows set by `MAX_ROWS_TO_RUN` in the setup cell.

For example, if the setup cell says:

`MAX_ROWS_TO_RUN = 5`

then this classification step will only run on the first 5 rows.

This is intentional for the first test run. Starting with a small number of rows makes it easier to check the prompt, review the outputs, catch errors, and avoid unnecessary API calls.

To run more rows, update `MAX_ROWS_TO_RUN` in the setup cell, then rerun:

1. The setup cell
2. The row context cell
3. This classification cell

For the sample dataset, you can set `MAX_ROWS_TO_RUN = 30` to process all rows.

In [ ]:
# ============================================================
# Run publication type classification on prototype rows
# ============================================================

import time

# ------------------------------------------------------------
# 1. Helper function to classify one publication row
# ------------------------------------------------------------
def classify_publication_row(row_payload):
    """
    Classify one publication row using the configured AI model.

    Returns a dictionary with:
    - row identifiers
    - parsed classification fields
    - validation/debugging fields
    """
    row_id = row_payload["row_id"]
    source_index = row_payload["source_index"]
    prompt = build_publication_prompt(row_payload)

    # --------------------------------------------------------
    # Call model through reusable helper
    # --------------------------------------------------------
    raw_output, model_call_success, model_error = call_ai_model(
        instructions=TASK_INSTRUCTIONS,
        prompt=prompt,
        max_output_tokens=500,
        temperature=0,
    )

    # --------------------------------------------------------
    # Parse model output only if the model call succeeded
    # --------------------------------------------------------
    if model_call_success:
        try:
            parsed_output = parse_model_json_response(raw_output)
            parse_success = True
            parse_error = ""

        except Exception as e:
            parsed_output = {}
            parse_success = False
            parse_error = str(e)

    else:
        parsed_output = {}
        parse_success = False
        parse_error = "Model call failed, so JSON parsing was skipped."

    # --------------------------------------------------------
    # Validate parsed output
    # --------------------------------------------------------
    if parse_success:
        validation_errors = validate_classification_output(parsed_output)
    else:
        validation_errors = ["Could not validate because JSON parsing failed."]

    # --------------------------------------------------------
    # Pull selected fields safely
    # --------------------------------------------------------
    publication_type = parsed_output.get("publication_type", "")
    confidence = parsed_output.get("confidence", "")
    rationale = parsed_output.get("rationale", "")

    # --------------------------------------------------------
    # Human review flag for prototype workflow
    # --------------------------------------------------------
    needs_human_review = (
        not model_call_success
        or not parse_success
        or len(validation_errors) > 0
        or confidence == "Low"
    )

    return {
        "PMID": row_id,
        "source_index": source_index,
        "publication_type": publication_type,
        "confidence": confidence,
        "rationale": rationale,
        "needs_human_review": needs_human_review,
        "model_call_success": model_call_success,
        "model_error": model_error,
        "parse_success": parse_success,
        "parse_error": parse_error,
        "validation_errors": "; ".join(validation_errors),
        "raw_model_output": raw_output,
    }


# ------------------------------------------------------------
# 2. Run classification across prototype rows
# ------------------------------------------------------------
classification_results = []

print(f"Running classification for {len(row_contexts)} prototype rows...\n")

for i, row_payload in enumerate(row_contexts, start=1):
    print(f"Processing row {i} of {len(row_contexts)} | PMID: {row_payload['row_id']}")

    result = classify_publication_row(row_payload)
    classification_results.append(result)

    # Small pause to be gentle on the API during workshop use
    time.sleep(0.25)

print("\nClassification run complete.")


# ------------------------------------------------------------
# 3. Convert results to dataframe
# ------------------------------------------------------------
results_df = pd.DataFrame(classification_results)

# ------------------------------------------------------------
# 4. Display review-friendly output
# ------------------------------------------------------------
review_columns = [
    "PMID",
    "publication_type",
    "confidence",
    "rationale",
    "needs_human_review",
    "parse_success",
    "validation_errors",
]

print("\nReview-friendly results:")
display(results_df[review_columns])

# ------------------------------------------------------------
# 5. Quick summary
# ------------------------------------------------------------
print("\nPublication type counts:")
display(results_df["publication_type"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "publication_type", "publication_type": "count"}
))

print("\nConfidence counts:")
display(results_df["confidence"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "confidence", "confidence": "count"}
))

print("\nRows needing human review:")
display(results_df[results_df["needs_human_review"] == True][review_columns])

## Sanity Check: What Just Happened?

At this point, we have completed a small end-to-end prototype run of the publication type classification workflow.

Here is what the notebook has done so far:

1. Loaded the sample publication dataset.
2. Selected `PMID` as the row identifier.
3. Sent only the `Title` and `Abstract` fields to the LLM.
4. Loaded human-defined publication type definitions from a Markdown file.
5. Built task instructions that constrained the model to specific allowed labels.
6. Tested the model on one row first.
7. Ran the classification workflow across the prototype rows.
8. Parsed the model responses into structured fields.
9. Validated the outputs against the allowed publication types and confidence levels.
10. Flagged rows for human review when the output was low confidence or failed validation.

This is the basic spec-driven workflow pattern:

**structured input → bounded LLM task → structured output → validation → human review**

The model is not making an open-ended judgment.  
It is making a constrained classification based on definitions, allowed labels, and a required output format.

## Things to Review Before Moving Forward

Before saving or scaling the workflow, review the results and ask:

- Do the publication type labels seem reasonable?
- Does the rationale appear appropriate based on the title and abstract?
- Are low-confidence rows appropriately flagged?
- Are any labels being overused or underused?
- Are the definitions specific enough to guide the model?
- Are any rows ambiguous enough to require human review?

If the outputs do not look right, revise the task instructions or category definitions before running more rows.

## Potential Next Steps

This prototype can be extended in several ways. Check out the ideas below! You can also continue along with the notebook! 

### Option 1: Scale beyond the prototype rows

Once the prompt, definitions, and validation checks look reasonable, increase `MAX_ROWS_TO_RUN` in the setup cell.

For the sample dataset, setting `MAX_ROWS_TO_RUN = 30` will process all rows.

For larger datasets, keep testing in small batches before running the full file.

### Option 2: Add another LLM prompt

A second model call could evaluate another bounded question, such as:

- Does this publication appear to involve empirical data?
- What type of data is collected in this paper?
- Does this publication appear to reuse an existing dataset?
- Does this publication mention a dataset, cohort, repository, registry, or trial?
- Is this publication relevant to a specific research theme?
- Does this publication contain signals that it may have associated data worth steward review?

This would create a multi-step workflow where each prompt performs one focused task.

### Option 3: Add stronger safety nets

The current prototype includes basic parsing and validation. A more robust version could add:

- Retry logic when the model call fails
- Retry logic when JSON parsing fails
- A fallback response when the model output is malformed
- More detailed logging of prompts, raw outputs, errors, and timestamps
- A maximum retry limit so the notebook does not get stuck
- Validation checks for missing fields, invalid labels, and blank rationales

These safety nets help prevent one bad row or one malformed model response from breaking the full workflow.

### Option 4: Add human review fields

The output could include additional columns for reviewer decisions, such as:

- `reviewer_publication_type`
- `reviewer_notes`
- `final_decision`
- `adjudication_status`

This would make the results easier to inspect, correct, and reuse later.

### Option 5: Retrieve richer publication text using DOI or PubMed/PMC links

You can optionally create a free **NCBI API key** to increase PubMed/PMC request limits when experimenting with DOI-based publication retrieval workflows. With this enhancement, the script can use the DOI field to look up PubMed records, check whether a linked PubMed Central (PMC) full-text version is available, and retrieve additional article text when access and reuse permissions allow.

### Option 6: Compare Different AI Models
Try changing the model to compare how responses, classifications, confidence levels, or reasoning may differ across models.

### Option 7: Explore Your Own Dataset

At this point, feel free to get creative and experiment with your own dataset using this workflow as a starting point. Think about tasks involving categorization, extraction, semantic matching, labeling, or structured review, then adapt the prompts, labels, reference materials, and decision rules to fit your use case.

## Merge LLM Results Back to the Original Dataset

The model responses have been parsed into a separate results dataframe called `results_df`.

In this step, we will merge those results back onto the original publication dataset so each processed row includes both:

1. The original publication metadata
2. The structured LLM classification output

The `source_index` column is used to reconnect each LLM result to the original row from the dataframe.

This creates a review-ready dataframe with new columns such as:

- `llm_publication_type`
- `llm_confidence`
- `llm_rationale`
- `llm_needs_human_review`
- `llm_parse_success`
- `llm_validation_errors`

Rows that were not included in the prototype run will remain blank in the new LLM output columns.

In [ ]:
# ============================================================
# Merge LLM results back to the original dataset
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required dataframes exist
# ------------------------------------------------------------
if "df_work" not in globals():
    raise NameError("df_work was not found. Please rerun the data preparation cell.")

if "results_df" not in globals():
    raise NameError("results_df was not found. Please rerun the classification cell.")

# ------------------------------------------------------------
# 2. Confirm required result columns exist
# ------------------------------------------------------------
required_results_columns = [
    "source_index",
    "publication_type",
    "confidence",
    "rationale",
    "needs_human_review",
    "parse_success",
    "validation_errors",
    "raw_model_output",
]

missing_result_columns = [
    col for col in required_results_columns
    if col not in results_df.columns
]

if missing_result_columns:
    raise ValueError(
        "The following required columns are missing from results_df:\n"
        + "\n".join(f"- {col}" for col in missing_result_columns)
    )

# ------------------------------------------------------------
# 3. Select and rename LLM output columns
# ------------------------------------------------------------
results_for_merge = results_df[
    [
        "source_index",
        "publication_type",
        "confidence",
        "rationale",
        "needs_human_review",
        "parse_success",
        "validation_errors",
        "raw_model_output",
    ]
].copy()

results_for_merge = results_for_merge.rename(
    columns={
        "publication_type": "llm_publication_type",
        "confidence": "llm_confidence",
        "rationale": "llm_rationale",
        "needs_human_review": "llm_needs_human_review",
        "parse_success": "llm_parse_success",
        "validation_errors": "llm_validation_errors",
        "raw_model_output": "llm_raw_model_output",
    }
)

# ------------------------------------------------------------
# 4. Merge results back to original working dataframe
# ------------------------------------------------------------
# source_index was captured from the original dataframe index.
# Joining on source_index preserves traceability to the original row.
df_with_llm_results = df_work.join(
    results_for_merge.set_index("source_index"),
    how="left"
)

# ------------------------------------------------------------
# 5. Display review-ready columns
# ------------------------------------------------------------
review_ready_columns = [
    "PMID",
    "Title",
    "Abstract",
    "llm_publication_type",
    "llm_confidence",
    "llm_rationale",
    "llm_needs_human_review",
    "llm_parse_success",
    "llm_validation_errors",
]

available_review_columns = [
    col for col in review_ready_columns
    if col in df_with_llm_results.columns
]

print("LLM results merged back to original dataset.")
print(f"Original rows: {len(df_work)}")
print(f"Rows with LLM classification: {df_with_llm_results['llm_publication_type'].notna().sum()}")

display(df_with_llm_results[available_review_columns].head(MAX_ROWS_TO_RUN))

# ------------------------------------------------------------
# 6. Quick sanity summary
# ------------------------------------------------------------
print("\nMerged publication type counts:")
display(
    df_with_llm_results["llm_publication_type"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "llm_publication_type", "llm_publication_type": "count"})
)

print("\nMerged confidence counts:")
display(
    df_with_llm_results["llm_confidence"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "llm_confidence", "llm_confidence": "count"})
)

## Build the Existing Data Use Task Instructions and Row Prompt

Now we will define the second LLM task.

This second task uses the output from the first LLM call as additional context.  
For each processed publication, the model will receive:

- `Title`
- `Abstract`
- `llm_rationale` from the publication type classification step

The title and abstract remain the primary evidence.  
The prior LLM rationale is included only as supporting context and should not override the title or abstract.

The goal is to classify whether the publication appears to use pre-existing data for analysis, rather than collecting new original data specifically for the study.

This step demonstrates how a spec-driven workflow can chain multiple bounded LLM calls together, where the structured output from one step becomes structured input for the next step.

In [ ]:
# ============================================================
# Build existing data use task instructions and row prompt
# ============================================================

# ------------------------------------------------------------
# 1. Define allowed labels for the second task
# ------------------------------------------------------------
ALLOWED_EXISTING_DATA_USE_LABELS = [
    "Likely used existing data",
    "Possibly used existing data",
    "Likely collected original data",
    "Unclear",
]

# Reuse the existing confidence levels from the first task:
# ALLOWED_CONFIDENCE_LEVELS = ["High", "Medium", "Low"]

# ------------------------------------------------------------
# 2. Define the expected JSON output structure
# ------------------------------------------------------------
EXISTING_DATA_EXPECTED_OUTPUT_FIELDS = {
    "existing_data_use_label": "One allowed label describing whether the publication appears to use pre-existing data.",
    "confidence": "High, Medium, or Low confidence in the existing data use classification.",
    "rationale": "A brief explanation based on the title, abstract, and prior LLM rationale."
}

# ------------------------------------------------------------
# 3. Build task instructions for the second LLM call
# ------------------------------------------------------------
EXISTING_DATA_TASK_INSTRUCTIONS = f"""
You are assisting with a publication review task.

Your job is to determine whether a publication appears to use pre-existing data for analysis.

In this task, "existing data use" means the publication appears to analyze data that had already
been collected before this publication's analysis, rather than collecting new original data specifically
for the study.

Use the title and abstract as the primary evidence.
You may use the prior LLM rationale as supporting context, but do not rely on it if it conflicts
with the title or abstract.

Do not use outside knowledge.
Do not infer details that are not supported by the title, abstract, or prior rationale.

Allowed existing data use labels:
{json.dumps(ALLOWED_EXISTING_DATA_USE_LABELS, indent=2)}

Label definitions:

- "Likely used existing data"
  Use this when the title or abstract clearly indicates use of a pre-existing dataset, database,
  registry, repository, cohort, trial dataset, electronic health record dataset, claims dataset,
  survey dataset, or secondary analysis.

- "Possibly used existing data"
  Use this when the title or abstract suggests possible use of pre-existing data, but the evidence
  is not fully clear. For example, the text may mention a cohort, database, records, registry,
  or previously collected data source, but it is ambiguous whether the authors collected the data
  themselves or reused existing data.

- "Likely collected original data"
  Use this when the title or abstract suggests that the study collected new data directly for this
  research. Examples may include recruitment, interviews, surveys, experiments, clinical assessments,
  participant enrollment, biological sample collection, or prospective data collection performed
  as part of the study.

- "Unclear"
  Use this when the title and abstract do not provide enough information to determine whether
  the study used existing data or collected original data.

Allowed confidence levels:
{json.dumps(ALLOWED_CONFIDENCE_LEVELS, indent=2)}

Confidence guidance:
- Use "High" when the title and abstract provide clear evidence for the selected label.
- Use "Medium" when the title and abstract provide some evidence, but there is some ambiguity.
- Use "Low" when the title and abstract provide weak, vague, missing, or conflicting evidence.

Return your answer as a single valid JSON object with exactly these fields:
{{
  "existing_data_use_label": "...",
  "confidence": "...",
  "rationale": "..."
}}

Do not include markdown.
Do not include extra text before or after the JSON.
"""

# ------------------------------------------------------------
# 4. Build the second row-level prompt
# ------------------------------------------------------------
def build_existing_data_use_prompt(row):
    """
    Build the user-facing prompt for the second LLM call.

    The PMID is retained outside the model call for merging results later.
    The model receives Title, Abstract, and the first LLM rationale.
    """
    title = clean_cell_value(row.get("Title", ""))
    abstract = clean_cell_value(row.get("Abstract", ""))
    prior_rationale = clean_cell_value(row.get("llm_rationale", ""))

    prompt = f"""
Determine whether this publication appears to use pre-existing data for analysis.

Title:
{title}

Abstract:
{abstract}

Prior LLM rationale from publication type classification:
{prior_rationale}
"""

    return prompt.strip()

# ------------------------------------------------------------
# 5. Create row payloads for the second task
# ------------------------------------------------------------
if "df_with_llm_results" not in globals():
    raise NameError(
        "df_with_llm_results was not found. Please rerun the merge cell before building the second task."
    )

required_second_task_columns = [
    "PMID",
    "Title",
    "Abstract",
    "llm_rationale",
]

missing_second_task_columns = [
    col for col in required_second_task_columns
    if col not in df_with_llm_results.columns
]

if missing_second_task_columns:
    raise ValueError(
        "The following required columns are missing from df_with_llm_results:\n"
        + "\n".join(f"- {col}" for col in missing_second_task_columns)
    )

# Only run the second task on rows that already have a first-call rationale.
second_task_df = df_with_llm_results[
    df_with_llm_results["llm_rationale"].notna()
].head(MAX_ROWS_TO_RUN).copy()

second_task_row_contexts = []

for source_index, row in second_task_df.iterrows():
    second_task_row_contexts.append(
        {
            "source_index": source_index,
            "row_id": clean_cell_value(row["PMID"]),
            "prompt": build_existing_data_use_prompt(row),
        }
    )

if not second_task_row_contexts:
    raise ValueError(
        "No rows were available for the second task. "
        "Make sure the first LLM call ran successfully and produced llm_rationale values."
    )

# ------------------------------------------------------------
# 6. Preview the second task instructions and first row prompt
# ------------------------------------------------------------
example_second_task_row = second_task_row_contexts[0]

print("Existing data use task instructions created successfully.")
print(f"Instruction length: {len(EXISTING_DATA_TASK_INSTRUCTIONS):,} characters")

print("\nAllowed existing data use labels:")
for label in ALLOWED_EXISTING_DATA_USE_LABELS:
    print(f"  - {label}")

print("\nExpected output fields:")
for field, description in EXISTING_DATA_EXPECTED_OUTPUT_FIELDS.items():
    print(f"  - {field}: {description}")

print(f"\nSecond task rows prepared: {len(second_task_row_contexts)}")

print("\nExample row ID retained outside model call:")
print(example_second_task_row["row_id"])

print("\nExample prompt sent to model for second task:")
print("-" * 80)
print(example_second_task_row["prompt"])
print("-" * 80)

## Test the Second LLM Call on One Publication

Before running the existing data use classification across multiple rows, we will test the second LLM call on one publication.

This cell sends one row to the model using:

- `Title`
- `Abstract`
- `llm_rationale` from the publication type classification step

The model will classify whether the publication appears to use pre-existing data for analysis.

This test checks that:

- The second model call works
- The response can be parsed as JSON
- The existing data use label is one of the allowed labels
- The confidence value is one of the allowed confidence levels
- The rationale field is present

Testing one row first helps catch prompt, parsing, or validation issues before running the full prototype set.

In [ ]:
# ============================================================
# Test second LLM call on one publication row
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required objects exist
# ------------------------------------------------------------
required_objects = [
    "second_task_row_contexts",
    "EXISTING_DATA_TASK_INSTRUCTIONS",
    "ALLOWED_EXISTING_DATA_USE_LABELS",
    "ALLOWED_CONFIDENCE_LEVELS",
    "parse_model_json_response",
    "call_ai_model",
]

for obj_name in required_objects:
    if obj_name not in globals():
        raise NameError(
            f"{obj_name} was not found. Please rerun the previous cells before testing the second LLM call."
        )

if not second_task_row_contexts:
    raise ValueError(
        "second_task_row_contexts is empty. Please rerun the cell that builds the second task row prompts."
    )


# ------------------------------------------------------------
# 2. Helper function to validate second task output
# ------------------------------------------------------------
def validate_existing_data_use_output(parsed_output):
    """
    Check whether the parsed model output follows the expected structure
    and uses only allowed labels for the existing data use task.
    """
    validation_errors = []

    required_fields = [
        "existing_data_use_label",
        "confidence",
        "rationale",
    ]

    for field in required_fields:
        if field not in parsed_output:
            validation_errors.append(f"Missing required field: {field}")

    existing_data_use_label = parsed_output.get("existing_data_use_label")
    confidence = parsed_output.get("confidence")
    rationale = parsed_output.get("rationale")

    if existing_data_use_label not in ALLOWED_EXISTING_DATA_USE_LABELS:
        validation_errors.append(
            f"Invalid existing_data_use_label: {existing_data_use_label}"
        )

    if confidence not in ALLOWED_CONFIDENCE_LEVELS:
        validation_errors.append(
            f"Invalid confidence: {confidence}"
        )

    if rationale is None or str(rationale).strip() == "":
        validation_errors.append("Rationale is missing or blank.")

    return validation_errors


# ------------------------------------------------------------
# 3. Select one row for testing
# ------------------------------------------------------------
second_test_row = second_task_row_contexts[0]
second_test_prompt = second_test_row["prompt"]

print(f"Testing second task row ID: {second_test_row['row_id']}")


# ------------------------------------------------------------
# 4. Call the model using the reusable helper
# ------------------------------------------------------------
second_raw_output, second_model_call_success, second_model_error = call_ai_model(
    instructions=EXISTING_DATA_TASK_INSTRUCTIONS,
    prompt=second_test_prompt,
    max_output_tokens=500,
    temperature=0,
)

if not second_model_call_success:
    raise RuntimeError(
        "The second model call failed. Check the API connection, model name, and prompt.\n"
        f"Model error: {second_model_error}"
    )


# ------------------------------------------------------------
# 5. Parse and validate the response
# ------------------------------------------------------------
try:
    second_parsed_output = parse_model_json_response(second_raw_output)
    second_parse_success = True
    second_parse_error = ""

except Exception as e:
    second_parsed_output = {}
    second_parse_success = False
    second_parse_error = str(e)

second_validation_errors = []

if second_parse_success:
    second_validation_errors = validate_existing_data_use_output(second_parsed_output)


# ------------------------------------------------------------
# 6. Print results
# ------------------------------------------------------------
print("\nRaw model output:")
print(second_raw_output)

print("\nModel call success:")
print(second_model_call_success)

if second_model_error:
    print("\nModel error:")
    print(second_model_error)

print("\nParse success:")
print(second_parse_success)

if second_parse_error:
    print("\nParse error:")
    print(second_parse_error)

print("\nParsed output:")
print(json.dumps(second_parsed_output, indent=2, ensure_ascii=False))

print("\nValidation errors:")
if second_validation_errors:
    for error in second_validation_errors:
        print(f"  - {error}")
else:
    print("  None. Output passed validation.")


# ------------------------------------------------------------
# 7. Store test result for inspection
# ------------------------------------------------------------
second_task_test_result = {
    "PMID": second_test_row["row_id"],
    "raw_model_output": second_raw_output,
    "model_call_success": second_model_call_success,
    "model_error": second_model_error,
    "parse_success": second_parse_success,
    "parse_error": second_parse_error,
    "validation_errors": second_validation_errors,
    **second_parsed_output,
}

print("\nSecond task test result object:")
print(json.dumps(second_task_test_result, indent=2, ensure_ascii=False))

## Run Existing Data Use Classification on Prototype Rows

Now that the one-row test worked, we can run the second LLM task across the prototype rows.

For each processed publication, this cell will:

1. Send the `Title`, `Abstract`, and first-call `llm_rationale` to the model
2. Classify whether the publication appears to use pre-existing data
3. Parse the response as JSON
4. Validate the output against the allowed labels
5. Store the results in a second reviewable dataframe

The `PMID` is retained as the row identifier so these results can be merged back to the original dataset later.

### Prototype Row Limit

This step only runs on the rows included in `second_task_row_contexts`.

That list was created using the current `MAX_ROWS_TO_RUN` value and only includes rows that already have a first-call `llm_rationale`.

To run more rows, update `MAX_ROWS_TO_RUN` in the setup cell, then rerun:

1. The setup cell
2. The row context cell
3. The first publication type classification cells
4. The merge cell
5. The second task prompt-building cell
6. This classification cell

In [ ]:
# ============================================================
# Run existing data use classification on prototype rows
# ============================================================

import time

# ------------------------------------------------------------
# 1. Helper function to classify existing data use for one row
# ------------------------------------------------------------
def classify_existing_data_use_row(row_payload):
    """
    Classify whether one publication appears to use pre-existing data.

    Returns a dictionary with:
    - row identifiers
    - parsed classification fields
    - validation/debugging fields
    """
    row_id = row_payload["row_id"]
    source_index = row_payload["source_index"]
    prompt = row_payload["prompt"]

    # --------------------------------------------------------
    # Call model through reusable helper
    # --------------------------------------------------------
    raw_output, model_call_success, model_error = call_ai_model(
        instructions=EXISTING_DATA_TASK_INSTRUCTIONS,
        prompt=prompt,
        max_output_tokens=500,
        temperature=0,
    )

    # --------------------------------------------------------
    # Parse model output only if the model call succeeded
    # --------------------------------------------------------
    if model_call_success:
        try:
            parsed_output = parse_model_json_response(raw_output)
            parse_success = True
            parse_error = ""

        except Exception as e:
            parsed_output = {}
            parse_success = False
            parse_error = str(e)

    else:
        parsed_output = {}
        parse_success = False
        parse_error = "Model call failed, so JSON parsing was skipped."

    # --------------------------------------------------------
    # Validate parsed output
    # --------------------------------------------------------
    if parse_success:
        validation_errors = validate_existing_data_use_output(parsed_output)
    else:
        validation_errors = ["Could not validate because JSON parsing failed."]

    # --------------------------------------------------------
    # Pull selected fields safely
    # --------------------------------------------------------
    existing_data_use_label = parsed_output.get("existing_data_use_label", "")
    confidence = parsed_output.get("confidence", "")
    rationale = parsed_output.get("rationale", "")

    # --------------------------------------------------------
    # Human review flag for prototype workflow
    # --------------------------------------------------------
    needs_human_review = (
        not model_call_success
        or not parse_success
        or len(validation_errors) > 0
        or confidence == "Low"
        or existing_data_use_label in ["Possibly used existing data", "Unclear"]
    )

    return {
        "PMID": row_id,
        "source_index": source_index,
        "existing_data_use_label": existing_data_use_label,
        "existing_data_confidence": confidence,
        "existing_data_rationale": rationale,
        "existing_data_needs_human_review": needs_human_review,
        "existing_data_model_call_success": model_call_success,
        "existing_data_model_error": model_error,
        "existing_data_parse_success": parse_success,
        "existing_data_parse_error": parse_error,
        "existing_data_validation_errors": "; ".join(validation_errors),
        "existing_data_raw_model_output": raw_output,
    }


# ------------------------------------------------------------
# 2. Run classification across second-task prototype rows
# ------------------------------------------------------------
existing_data_results = []

print(f"Running existing data use classification for {len(second_task_row_contexts)} prototype rows...\n")

for i, row_payload in enumerate(second_task_row_contexts, start=1):
    print(f"Processing row {i} of {len(second_task_row_contexts)} | PMID: {row_payload['row_id']}")

    result = classify_existing_data_use_row(row_payload)
    existing_data_results.append(result)

    # Small pause to be gentle on the API during workshop use
    time.sleep(0.25)

print("\nExisting data use classification run complete.")


# ------------------------------------------------------------
# 3. Convert results to dataframe
# ------------------------------------------------------------
existing_data_results_df = pd.DataFrame(existing_data_results)

# ------------------------------------------------------------
# 4. Display review-friendly output
# ------------------------------------------------------------
existing_data_review_columns = [
    "PMID",
    "existing_data_use_label",
    "existing_data_confidence",
    "existing_data_rationale",
    "existing_data_needs_human_review",
    "existing_data_parse_success",
    "existing_data_validation_errors",
]

print("\nReview-friendly existing data use results:")
display(existing_data_results_df[existing_data_review_columns])

# ------------------------------------------------------------
# 5. Quick summary
# ------------------------------------------------------------
print("\nExisting data use label counts:")
display(
    existing_data_results_df["existing_data_use_label"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "existing_data_use_label", "existing_data_use_label": "count"})
)

print("\nExisting data use confidence counts:")
display(
    existing_data_results_df["existing_data_confidence"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "existing_data_confidence", "existing_data_confidence": "count"})
)

print("\nRows needing human review for existing data use:")
display(
    existing_data_results_df[
        existing_data_results_df["existing_data_needs_human_review"] == True
    ][existing_data_review_columns]
)

## Merge Existing Data Use Results Back to the Dataset

The second LLM call results have been parsed into a separate dataframe called `existing_data_results_df`.

In this step, we will merge those results back onto `df_with_llm_results`, which already contains:

- Original publication fields
- Publication type classification results from the first LLM call

After this merge, the final review dataframe will also include:

- `llm_existing_data_use_label`
- `llm_existing_data_confidence`
- `llm_existing_data_rationale`
- `llm_existing_data_needs_human_review`
- `llm_existing_data_parse_success`
- `llm_existing_data_validation_errors`

The `source_index` column is used again to reconnect the second LLM results to the correct original row.

Rows that were not included in the prototype run will remain blank in the second-task output columns.

In [ ]:
# ============================================================
# Merge existing data use results back to the dataset
# ============================================================

# ------------------------------------------------------------
# 1. Confirm required dataframes exist
# ------------------------------------------------------------
if "df_with_llm_results" not in globals():
    raise NameError(
        "df_with_llm_results was not found. Please rerun the first merge cell."
    )

if "existing_data_results_df" not in globals():
    raise NameError(
        "existing_data_results_df was not found. Please rerun the existing data use classification cell."
    )

# ------------------------------------------------------------
# 2. Confirm required result columns exist
# ------------------------------------------------------------
required_existing_data_columns = [
    "source_index",
    "existing_data_use_label",
    "existing_data_confidence",
    "existing_data_rationale",
    "existing_data_needs_human_review",
    "existing_data_parse_success",
    "existing_data_validation_errors",
    "existing_data_raw_model_output",
]

missing_existing_data_columns = [
    col for col in required_existing_data_columns
    if col not in existing_data_results_df.columns
]

if missing_existing_data_columns:
    raise ValueError(
        "The following required columns are missing from existing_data_results_df:\n"
        + "\n".join(f"- {col}" for col in missing_existing_data_columns)
    )

# ------------------------------------------------------------
# 3. Select and rename second-task LLM output columns
# ------------------------------------------------------------
existing_data_for_merge = existing_data_results_df[
    [
        "source_index",
        "existing_data_use_label",
        "existing_data_confidence",
        "existing_data_rationale",
        "existing_data_needs_human_review",
        "existing_data_parse_success",
        "existing_data_validation_errors",
        "existing_data_raw_model_output",
    ]
].copy()

existing_data_for_merge = existing_data_for_merge.rename(
    columns={
        "existing_data_use_label": "llm_existing_data_use_label",
        "existing_data_confidence": "llm_existing_data_confidence",
        "existing_data_rationale": "llm_existing_data_rationale",
        "existing_data_needs_human_review": "llm_existing_data_needs_human_review",
        "existing_data_parse_success": "llm_existing_data_parse_success",
        "existing_data_validation_errors": "llm_existing_data_validation_errors",
        "existing_data_raw_model_output": "llm_existing_data_raw_model_output",
    }
)

# ------------------------------------------------------------
# 4. Merge second-task results back to the existing merged dataframe
# ------------------------------------------------------------
final_review_df = df_with_llm_results.join(
    existing_data_for_merge.set_index("source_index"),
    how="left"
)

# ------------------------------------------------------------
# 5. Display final review-ready columns
# ------------------------------------------------------------
final_review_columns = [
    "PMID",
    "Title",
    "Abstract",
    "llm_publication_type",
    "llm_confidence",
    "llm_rationale",
    "llm_needs_human_review",
    "llm_existing_data_use_label",
    "llm_existing_data_confidence",
    "llm_existing_data_rationale",
    "llm_existing_data_needs_human_review",
    "llm_existing_data_parse_success",
    "llm_existing_data_validation_errors",
]

available_final_review_columns = [
    col for col in final_review_columns
    if col in final_review_df.columns
]

print("Existing data use results merged back to the dataset.")
print(f"Original rows: {len(df_work)}")
print(
    "Rows with publication type classification: "
    f"{final_review_df['llm_publication_type'].notna().sum()}"
)
print(
    "Rows with existing data use classification: "
    f"{final_review_df['llm_existing_data_use_label'].notna().sum()}"
)

display(final_review_df[available_final_review_columns].head(MAX_ROWS_TO_RUN))

# ------------------------------------------------------------
# 6. Quick final sanity summaries
# ------------------------------------------------------------
print("\nFinal publication type counts:")
display(
    final_review_df["llm_publication_type"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "llm_publication_type", "llm_publication_type": "count"})
)

print("\nFinal existing data use label counts:")
display(
    final_review_df["llm_existing_data_use_label"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "llm_existing_data_use_label", "llm_existing_data_use_label": "count"})
)

print("\nRows needing human review from either task:")
human_review_mask = (
    final_review_df["llm_needs_human_review"].fillna(False)
    | final_review_df["llm_existing_data_needs_human_review"].fillna(False)
)

display(final_review_df.loc[human_review_mask, available_final_review_columns])

## Export Final Review Outputs

Now that both LLM task outputs have been merged back onto the original publication dataset, we can export the results.

This export creates an Excel workbook with multiple sheets:

1. `review_output`
   - A clean, human-review-friendly sheet with the original publication fields and the main LLM outputs.

2. `all_outputs`
   - A fuller sheet with all available columns, including raw/debug fields.

3. `run_metadata`
   - A small summary of the notebook run, including model name, row count, timestamp, and task labels.

The exported file will be saved in the repo’s `outputs/` folder.

The output filename is based on the original input filename with `_output` and the run timestamp added before the file extension.

For example:

`publicationslist_HEAL.xlsx`

becomes something like:

`publicationslist_HEAL_output_20260527_143015.xlsx`

For a workshop or prototype setting, this gives each team a concrete file they can inspect, share, or continue refining without writing generated files back into the source `data/` folder.

In [ ]:
# ============================================================
# Export final review outputs
# ============================================================

# ------------------------------------------------------------
# 1. Confirm final dataframe exists
# ------------------------------------------------------------
if "final_review_df" not in globals():
    raise NameError(
        "final_review_df was not found. Please rerun the merge cell before exporting."
    )

# ------------------------------------------------------------
# 2. Define export file path
# ------------------------------------------------------------
# Save generated outputs in the repo's outputs folder.
# This keeps source data and generated results separate.
EXPORT_DIR = OUTPUT_DIR
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Use the original input file name and add "_output" plus the run timestamp.
# Example: publicationslist_HEAL.xlsx -> publicationslist_HEAL_output_20260527_143015.xlsx
export_file_name = f"{DATA_PATH.stem}_output_{RUN_TIMESTAMP}.xlsx"
export_path = EXPORT_DIR / export_file_name

# ------------------------------------------------------------
# 3. Define clean review columns
# ------------------------------------------------------------
review_output_columns = [
    "PMID",
    "PMCID",
    "DOI",
    "Title",
    "Abstract",
    "llm_publication_type",
    "llm_confidence",
    "llm_rationale",
    "llm_needs_human_review",
    "llm_parse_success",
    "llm_validation_errors",
    "llm_existing_data_use_label",
    "llm_existing_data_confidence",
    "llm_existing_data_rationale",
    "llm_existing_data_needs_human_review",
    "llm_existing_data_parse_success",
    "llm_existing_data_validation_errors",
]

available_review_output_columns = [
    col for col in review_output_columns
    if col in final_review_df.columns
]

review_output_df = final_review_df[available_review_output_columns].copy()

# ------------------------------------------------------------
# 4. Create run metadata sheet
# ------------------------------------------------------------
run_metadata = {
    "run_timestamp": RUN_TIMESTAMP,
    "model_name": MODEL_NAME,
    "data_mode": DATA_MODE,
    "data_path": str(DATA_PATH),
    "export_path": str(export_path),
    "max_rows_to_run": MAX_ROWS_TO_RUN,
    "total_rows_in_original_dataset": len(df_work),
    "rows_with_publication_type_classification": final_review_df["llm_publication_type"].notna().sum(),
    "rows_with_existing_data_use_classification": final_review_df["llm_existing_data_use_label"].notna().sum(),
    "publication_type_labels": "; ".join(ALLOWED_PUBLICATION_TYPES),
    "existing_data_use_labels": "; ".join(ALLOWED_EXISTING_DATA_USE_LABELS),
    "confidence_levels": "; ".join(ALLOWED_CONFIDENCE_LEVELS),
}

run_metadata_df = pd.DataFrame(
    list(run_metadata.items()),
    columns=["metadata_field", "metadata_value"]
)

# ------------------------------------------------------------
# 5. Export to Excel workbook
# ------------------------------------------------------------
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    review_output_df.to_excel(
        writer,
        sheet_name="review_output",
        index=False
    )

    final_review_df.to_excel(
        writer,
        sheet_name="all_outputs",
        index=False
    )

    run_metadata_df.to_excel(
        writer,
        sheet_name="run_metadata",
        index=False
    )

# ------------------------------------------------------------
# 6. Print export summary
# ------------------------------------------------------------
print("Export complete.")
print(f"Input file:  {DATA_PATH}")
print(f"Output file: {export_path.resolve()}")

print("\nSheets included:")
print("  - review_output")
print("  - all_outputs")
print("  - run_metadata")

print("\nReview output preview:")
display(review_output_df.head(MAX_ROWS_TO_RUN))

## Congratulations: You Built a Two-Step Spec-Driven LLM Workflow

Congratulations! You have now built a small, end-to-end spec-driven LLM workflow.

In this notebook, you loaded a structured dataset, selected specific columns for model review, defined bounded classification tasks, called the LLM row by row, parsed structured JSON responses, validated model outputs, merged the results back onto the original dataset, and exported a review-ready file.

You also demonstrated a chained LLM workflow. The first model call completed one structured review task, and the second model call used the original row context plus the first-call output to complete a follow-up evaluation.

The overall workflow pattern was:

**structured input → bounded prompt → structured output → validation → merge → review-ready export**

This is the core idea behind a spec-driven workflow.

The model is not being asked to “just decide.” It is being given a defined task, allowed labels, expected fields, evidence boundaries, and review criteria. Python handles the repeatable workflow steps, while the LLM performs a bounded interpretation task inside the structure you define.

## Recommended Additional Tasks

If you want to keep building on this prototype, here are some possible next steps.

### 1. Pull richer metadata from external APIs

Extend the workflow by pulling additional metadata from public or approved APIs.

Depending on your dataset, this could include:

- Publication metadata
- Grant or award metadata
- Study metadata
- Repository metadata
- Clinical trial metadata
- Organization or affiliation metadata
- Dates, identifiers, keywords, or subject terms

For publication-focused workflows, APIs such as PubMed, PubMed Central, Crossref, OpenAlex, Semantic Scholar, NIH RePORTER, or publisher APIs may provide additional context.

Important note: Some APIs provide citation or summary metadata only, while others may provide abstracts, full text, links, or structured records. Always check access, reuse permissions, and rate limits before expanding the workflow.

### 2. Match records across data sources

Add a matching step to connect records from one dataset to another.

Possible matching targets could include:

- Grant numbers
- DOIs
- PMIDs
- Trial registration numbers
- Repository accession IDs
- Organization names
- Project IDs
- Dataset identifiers
- Internal system IDs

This kind of step can help connect scattered information across spreadsheets, APIs, databases, or public records.

### 3. Add another bounded classification task

Create a new LLM task with its own definitions, allowed labels, rationale field, and confidence score.

Examples might include:

- Is this record relevant to a specific research topic?
- Does this project involve human subjects, animal models, or computational methods?
- Does this publication describe original data collection or secondary analysis?
- Does this abstract mention a specific population, intervention, method, or outcome?
- Does this record require follow-up review?
- Does this row appear complete, ambiguous, or inconsistent?

The key is to define the task clearly before asking the model to evaluate each row.

### 4. Look for reuse, quality, or review signals

Add a more detailed prompt to identify signals that may support review, curation, prioritization, or reuse.

Depending on your use case, these signals could include:

- Named datasets
- Cohorts
- Registries
- Repositories
- Data sharing statements
- Public-use datasets
- Controlled-access datasets
- Secondary analysis language
- Missing metadata
- Ambiguous labels
- Inconsistent terminology
- Evidence of standardization
- Fields needing human review

This can help transform a spreadsheet into a more structured review queue.

### 5. Try the workflow with another dataset

The same pattern can be adapted to many structured datasets, such as:

- Publication lists
- Grant or award records
- ClinicalTrials.gov records
- Repository metadata
- Survey metadata
- Data dictionaries
- Issue trackers
- Monday.com board exports
- Airtable exports
- Customer feedback
- Meeting notes
- Support tickets
- Internal project inventories

Bring your own dataset, bring your own labels, bring your own tiny research goblin.

The workflow stays the same:

**define the task → constrain the model → validate the output → keep a human in the loop**

## Final Reflection

Before turning any prototype into a larger workflow, ask:

- What decision is the model making?
- Are the allowed labels clear?
- What evidence should the model use?
- What evidence should the model ignore?
- What fields should the model return?
- What outputs need validation?
- Which rows require human review?
- What should happen when the model is uncertain?
- Which parts of the workflow should be deterministic Python logic?
- Which parts actually benefit from LLM interpretation?
- Would this remain a structured workflow, or should it eventually become something more agentic?

The goal is not to replace expert judgment.

The goal is to make repetitive review tasks more structured, transparent, auditable, and easier to improve.